# The average that lies

**Week 1, Monday. Notebook 4 of 4.** By the end of this notebook you can compute a mean and a
median by hand, say why one record moves the first and leaves the second, and choose between them
on purpose, which is what Anand meant by "no averages".

> **The client asks.** "No averages. One business customer can move an average."
>
> Anand Iyer, finance controller, Kalpa Retail

Notebook 3 ended on revenue per order, Rs 18,160. That number is a mean, and this notebook asks
the question a finance controller asks of every mean: which order does it describe?

**Setup.** Find the helper, load the orders, and build `amounts`, the thirty order values as
numbers, with `int()` doing the morning's patch. Nothing here prints the list: reading it is your
job, further down.

In [1]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

ORDERS = kit.load_records("C2_W01_D01_orders_STUDENT.py")
amounts = []
for order in ORDERS:
    amounts.append(int(order["amount"]))
print(len(amounts), "amounts ready")

30 amounts ready


In [2]:
kit.side_by_side(
    kit.ladder(["The question before the budget", "The first count", "The leaves", "The average that lies", "Hands-on: your call"], lit=3, show=False),
    kit.vflow(["the mean\nthe total spread evenly",
               "the median\nthe middle of the sorted list",
               "your turn\nread the top of the list",
               "the mechanism\non five invented orders",
               "the choice\nwhich number for which job"], show=False),
)

## 1. Revenue per order is a mean

The mean spreads the total evenly over the orders: Rs 5,44,810 over 30 orders is Rs 18,160. Put
that in Meera's note as "the typical order" and it says Kalpa Retail sells eighteen-thousand-rupee
baskets of consumer goods.

**Predict before you run.** Does Rs 18,160 describe a typical order? a) yes, it is the average of
every order; b) no, most orders sit far below it; c) no, most orders sit far above it; d) it
cannot be judged without the segment.

The median settles it: sort the amounts and take the one in the middle. With an even count there
are two in the middle, positions 15 and 16 of 30, and the median is their average.

In [3]:
mean = sum(amounts) / len(amounts)

ordered = sorted(amounts)
n = len(ordered)
low_middle = ordered[n // 2 - 1]      # position 15 of 30, index 14
high_middle = ordered[n // 2]         # position 16 of 30, index 15
median = (low_middle + high_middle) / 2

print(f"mean   {mean:,.2f}")
print(f"middle {low_middle} and {high_middle}")
print(f"median {median:,.1f}")
kit.stats([(kit.rupees(mean), "the mean", "revenue over orders"),
           (kit.rupees(median), "the median", "the middle of 30 sorted amounts"),
           (f"{mean / median:.1f}", "times", "the mean over the median")])

mean   18,160.33
middle 2110 and 2300
median 2,205.0


**What happened.** The answer is b. The middle order is Rs 2,205, and the mean sits about eight
times higher, above almost every order in the file. Something at the top of the sorted list is
pulling it up.

**Your turn.** Find out what. Type these lines into the empty cell below and run them, then say in
one sentence what sits at the top of the list and what kind of order it must be:

```python
print(ordered[-5:])                  # the five largest amounts
for order in ORDERS:
    if int(order["amount"]) == ordered[-1]:
        print(order)                 # the whole record behind the largest one
kit.strip(ordered, markers=[("mean", mean, "bad"), ("median", median, "good")],
          title="All thirty orders on one axis")
```

The sentence you just wrote is Anand's warning, proved on Kalpa's own file. Before deciding what to
do about it, the next section shows the mechanism on numbers nobody has to trust.

## 2. The mechanism, on five invented orders

Five ordinary orders, then the same five with one bulk order in place of the largest.

**Predict before you run.** When Rs 2,600 becomes Rs 90,000, roughly how far does the mean move,
and how far does the median move?

In [4]:
ordinary = [1900, 2100, 2200, 2400, 2600]
with_bulk = [1900, 2100, 2200, 2400, 90000]

mean_1 = sum(ordinary) / 5
mean_2 = sum(with_bulk) / 5
median_1 = sorted(ordinary)[2]
median_2 = sorted(with_bulk)[2]

kit.strip(ordinary, markers=[("mean", mean_1, "bad"), ("median", median_1, "good")],
          lo=0, hi=100000, title="Five ordinary orders: the mean and the median sit together")
kit.strip(with_bulk, markers=[("mean", mean_2, "bad"), ("median", median_2, "good")],
          lo=0, hi=100000, lit=[4], title="One bulk order joins: the mean leaves the crowd, the median stays")
kit.table(["", "mean", "median"],
          [("five ordinary orders", kit.rupees(mean_1), kit.rupees(median_1)),
           ("one bulk order in place of the largest", kit.rupees(mean_2), kit.rupees(median_2))],
          caption="One record changed out of five")

,mean,median
five ordinary orders,"Rs 2,240","Rs 2,200"
one bulk order in place of the largest,"Rs 19,720","Rs 2,200"


**What happened.** The mean moved from Rs 2,240 to Rs 19,720, almost nine times; the median did not
move at all. Two ways to hold this in your head:

| Picture | The mean | The median |
|---|---|---|
| A see-saw | The balance point: one heavy weight far out tips it all the way | Ignores how far out the weight sits |
| A queue sorted by amount | Needs every value's size | Only asks who stands in the middle, and a giant joining the end does not change that |

**Rigor: by exactly how much does the mean move?** Change one value by an amount d and the total
changes by d, so the mean changes by d divided by n. Here d is Rs 87,400 and n is 5, so the mean
rises by Rs 17,480, from Rs 2,240 to Rs 19,720. The median moves only if the change carries a value
across the middle of the sorted list, which a bigger largest value never does.

In [5]:
d = 90000 - 2600
kit.check("the mean moves by d over n", mean_2 - mean_1 == d / 5,
          f"moved {mean_2 - mean_1:,.0f}; d / n is {d / 5:,.0f}")
kit.check("the median does not move", median_1 == median_2 == 2200, f"{median_1} and {median_2}")
kit.check("on the real file the mean sits more than eight times the median", mean / median > 8,
          f"{mean / median:.2f}")

## 3. The median by hand, with both cases

| Count | How many middles | The median | For thirty orders |
|---|---|---|---|
| Odd | One | `ordered[n // 2]` | Not this case |
| Even | Two | `(ordered[n // 2 - 1] + ordered[n // 2]) / 2` | Indexes 14 and 15, which are positions 15 and 16 |

Python counts from zero and people count from one, so index 14 is the fifteenth value. That
off-by-one is the most common median bug, and it is why the cell below checks both middles.

In [6]:
kit.flow(["sort the 30 amounts\nsmallest first",
          f"the middle two\npositions 15 and 16",
          f"their values\n{low_middle:,} and {high_middle:,}",
          f"their average\n{kit.rupees(median)}"],
         lit=3, title="The median of an even count, step by step")
odd = sorted([940, 1190, 2060, 2110, 2300])
print(odd[len(odd) // 2])      # the odd case: one middle, at index n // 2

2060


In [7]:
kit.check("the median of thirty orders is Rs 2,205", median == 2205, f"{median}")
kit.check("the two middles are the 15th and 16th values", (low_middle, high_middle) == (2110, 2300),
          f"{low_middle} and {high_middle}")
kit.check("the odd case takes the single middle value", odd[len(odd) // 2] == 2060, str(odd))

## 4. Which number goes in Meera's note?

The note needs one number for a typical order, and a decision about the top of the list. Choose:

a) the mean, Rs 18,160, because it uses every order; b) the median, Rs 2,205, with the largest order
named on its own line; c) the mean after deleting the largest order; d) the largest order, because
it drives the revenue.

In [8]:
kit.tree({"label": "what is the number for?", "kind": "lit", "branches": [
    ("", {"label": "a typical order", "kind": "lit", "branches": [
        ("", {"label": "the median\none record cannot drag it", "kind": "good"})]}),
    ("", {"label": "a total that must add up", "branches": [
        ("", {"label": "the mean\nit multiplies back to the total"})]}),
    ("", {"label": "a first look at a file", "branches": [
        ("", {"label": "both\nthe gap is itself a finding"})]})]},
    title="The right average depends on the job it does; Meera's note needs the lit path")

**The answer is b.** The mean is not wrong; it answers a different question. Revenue per order on
the tree is a mean on purpose, because it has to multiply back to the total. The typical order is a
median on purpose, because it has to describe an order somebody placed.

> **Kavya's review.** Option c is the dangerous one, because that order is real revenue. Report the
> median as the typical order, name the large order on its own line, and keep the total whole.

## 5. In the interview

**[S] Mean or median for order value, and why?** "It depends on what the number is for. Order
values are usually skewed: a few very large orders sit far above the rest, and the mean is pulled
toward them while the median is not. For the typical order I report the median. For anything that
has to multiply back to a total, like revenue per order on a driver tree, I use the mean, because a
median does not reconcile. On a first look at a file I report both, since a wide gap between them
is itself a finding." The interviewer is listening for the dependence on purpose, the word skew,
and the reconciliation argument.

**[F] A business says "grow revenue 15 percent"; how do you turn that into questions data can
answer?** "Five moves. Restate the goal with its window and its baseline: 15 percent over which
period, from what number. Break revenue into its drivers on a tree. For each driver, name the data
that measures it, with its denominator. Size the gap per driver: which one would have to move, and
by how much, to deliver 15 percent. Then say which decision each answer would change, so the
analysis ends in a choice and not in a report." The interviewer is listening for a baseline, a
decomposition, and a link to a decision.

The other three questions of the day are answered where they arose: [S] increasing sales and [D]
the acquisition budget in notebook 1, and [SV] lists against dictionaries in notebook 2.

In [9]:
kit.table(["The number", "Its definition", "Value"],
          [("Revenue", "all booked orders, 1 July to 26 September", kit.rupees(sum(amounts))),
           ("Customers", "distinct customer ids", "23"),
           ("Orders per customer", "30 orders over 23 customers", "1.30"),
           ("Revenue per order", "the mean: revenue over orders", kit.rupees(mean)),
           ("The typical order", "the median of 30 sorted amounts", kit.rupees(median))],
          caption="The day's numbers, each with its definition, ready for Meera's note")
kit.flow(["claim\nwhich branch to open first", "evidence\na number with its definition",
          "caveat\nwhat 30 orders cannot show", "next step\nwhat gets compared tomorrow"],
         title="The shape of every sentence to Meera this week")

The number,Its definition,Value
Revenue,"all booked orders, 1 July to 26 September","Rs 5,44,810"
Customers,distinct customer ids,23
Orders per customer,30 orders over 23 customers,1.30
Revenue per order,the mean: revenue over orders,"Rs 18,160"
The typical order,the median of 30 sorted amounts,"Rs 2,205"


In [10]:
kit.check_summary()
print("Next: the hands-on notebook, your call on delivered orders; then Tuesday, which asks which branch moved.")

Next: the hands-on notebook, your call on delivered orders; then Tuesday, which asks which branch moved.
